# Bias ordering check with $\rho_5$ (exploratory, not part of the plan)

**Question.** The density cells show that Fil-underdense keeps the BAO while NonFil-dense loses it.
That argues "BAO strength does not follow bias" **only if** NonFil-dense really has the larger
*large-scale* bias. Local density suggests so, but filament membership also raises bias, so the
ordering has to be measured, not assumed.

**Idea.** On scales where (i) bias is scale-independent and (ii) the BAO contributes negligibly,

$$\xi_A(s) = R\,\xi_B(s), \qquad R = (b_A/b_B)^2 .$$

So we estimate $R$ directly from the two measured monopoles: no template amplitude, no broadband,
no $B$. Only the *choice of scales* uses the linear template (to know where the BAO starts).

**Steps**
1. Catalogue and magnitude cut (same as `scripts/multidark_analysis.py`).
2. $\rho_5$ from the 5th-neighbour distance; quick comparison with $\rho_3$.
3. Subsamples: Full, Fil, NonFil and Fil/NonFil × dense/underdense in $\rho_5$.
4. Monopoles $\xi_0(s)$ with the pipeline functions (real space: box positions, no RSD).
5. Joint Gaussian covariance of all monopoles from their measured auto/cross $P(k)$.
6. Upper limit of the scale range: where the BAO part of the template becomes non-negligible.
7. Estimator for $R$ that never divides noisy numbers.
8. Lower limit: the plateau test (smallest $s_{\min}$ for which a constant $R$ fits).
9. Results: bias of every sample relative to Full, and the key pair NonFil-dense vs Fil-underdense.

Everything this notebook writes goes to `data/processed/exploratory/bias_ordering_rho5/`; it
does not touch the pipeline's folders.

In [ ]:
import os, sys, json
REPO = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))   # notebook lives in notebooks/exploratory
sys.path.insert(0, os.path.join(REPO, 'src'))
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats

from multidark_data_loader import load_catalog, select_sample, compute_knn_distance
from multidark_correlation import compute_xi_s_mu, compute_monopole_from_xi_s_mu
from multidark_power import measure_spectra
from bao_fitting import (linear_theory, generate_templates, BAOTemplate, sigma_nl_theory,
                         load_spectra, gaussian_cov_pk, estimate_bias_independent)

## 0. Settings

Binning, magnitude cut and the filament definition are the ones of run 1 of the fitting notebook
($d_{\rm fil}\le 3$ for Fil, $10\le d_{\rm fil}\le 100$ for NonFil). "Dense" = above the 84th
percentile of $\log\rho_5$, as in the $\rho_3$ configuration of the script.

`bao_tolerance` sets the upper scale limit (step 6) and `p_plateau` the lower one (step 8);
both are explained below. `S_MIN` / `S_MAX` = `None` means "use the value suggested by the data";
set a number to override.

In [ ]:
L = 1000.0                      # box side, h^-1 Mpc
mag_max = -21.2

min_sep, max_sep, bin_size = 5.0, 150.0, 5.0      # same s bins as the pipeline

fil_interval = (0.0, 3.0)       # d_fil, h^-1 Mpc   (inclusive, as split_by_variable)
nonfil_interval = (10.0, 100.0)

k_nn = 5                        # rho_k: density from the distance to the k-th neighbour
dense_percentile = 84           # dense = log rho_k above this percentile of the full sample

pk_ngrid = 384                  # FFT grid for the spectra (as the script)
force_recompute = False         # True: redo pair counts and spectra

bao_tolerance = 0.05            # step 6: max fractional BAO contribution allowed in a bin
p_plateau = 0.05                # step 8: a constant R is "acceptable" if its chi2 p-value > this
s_min_scan = [15, 20, 25, 30, 35, 40]
S_MIN = None                    # None -> suggested by the plateau test
S_MAX = None                    # None -> suggested by the template (step 6)

# Comparisons.  The first one is the one the density-cell argument depends on.
key_pairs = [('NonFil dense', 'Fil underdense'),
             ('Fil', 'NonFil')]
reference = 'Full'              # every sample is also compared with this one

OUT = os.path.join(REPO, 'data', 'processed', 'exploratory', f'bias_ordering_rho{k_nn}')
os.makedirs(OUT, exist_ok=True)
RAW = os.path.join(REPO, 'data', 'raw')

## 1. Catalogue

Same loader and cut as the script: positions in real space (the velocities are read but not used).

In [ ]:
cat = select_sample(load_catalog(os.path.join(RAW, 'to_mock.csv'),
                                 os.path.join(RAW, 'mock_withfilament.csv')), mag_max)
print(f"{len(cat)} galaxies with M_r < {mag_max}")

## 2. Local density $\rho_5$

$\rho_k = k \,/\, \big(\tfrac{4}{3}\pi d_k^3\big)$, with $d_k$ the distance to the $k$-th neighbour
(periodic box), exactly as the script does for $k=3$.

Why $k=5$ rather than 3:
* the estimate is less noisy: the relative Poisson scatter of $\rho_k$ goes roughly as $1/\sqrt{k}$
  (≈ 58 % → 45 %);
* the smoothing scale grows only as $k^{1/3}$ (≈ +19 %), so $\rho_5$ still measures *local*
  (group/halo-scale) density. It is not a large-scale-bias estimator, which is why we measure
  the bias ordering below instead of assuming it.

The cell prints the median $d_k$ (the actual smoothing scale) and how many galaxies change
"dense" label between $\rho_3$ and $\rho_5$, to see how different the two splits really are.

In [ ]:
pos = cat[['x', 'y', 'z']].values
d_k = compute_knn_distance(pos, k=k_nn, boxsize=L)
d_3 = compute_knn_distance(pos, k=3, boxsize=L)
cat[f'log_rho_{k_nn}'] = np.log10(k_nn / (4 * np.pi * d_k**3 / 3))
cat['log_rho_3'] = np.log10(3 / (4 * np.pi * d_3**3 / 3))

col = f'log_rho_{k_nn}'
thr_k = np.percentile(cat[col], dense_percentile)
thr_3 = np.percentile(cat['log_rho_3'], dense_percentile)
dense_k = cat[col].values > thr_k
dense_3 = cat['log_rho_3'].values > thr_3
print(f"median d_3 = {np.median(d_3):.2f}, median d_{k_nn} = {np.median(d_k):.2f} h^-1 Mpc")
print(f"Spearman(log rho_3, log rho_{k_nn}) = {stats.spearmanr(cat['log_rho_3'], cat[col])[0]:.3f}")
print(f"dense in rho_3 but not rho_{k_nn}: {np.mean(dense_3 & ~dense_k):.3%} of all galaxies; "
      f"the reverse: {np.mean(~dense_3 & dense_k):.3%}")

fig, ax = plt.subplots(figsize=(5, 4.5))
ax.hexbin(cat['log_rho_3'], cat[col], gridsize=80, bins='log', cmap='Greys')
ax.axvline(thr_3, color='tomato', lw=1); ax.axhline(thr_k, color='steelblue', lw=1)
ax.set_xlabel(r'$\log_{10}\rho_3$'); ax.set_ylabel(rf'$\log_{{10}}\rho_{k_nn}$')
ax.set_title(f'dense thresholds ({dense_percentile}th percentile)')
plt.show()

## 3. Subsamples

The percentile threshold is computed on the full magnitude-limited sample (as `split_by_variable`
does), so "dense" means the same density in Fil and NonFil. Dense/underdense is a strict
partition here (the script's inclusive intervals put the one galaxy at the threshold in both;
irrelevant).

The number densities differ a lot between samples. That changes the *errors* of $\xi$
(shot noise, included in the covariance), not its expected value, so $R$ is not biased by it.

In [ ]:
dfil = cat['dist_fil'].values
fil = (dfil >= fil_interval[0]) & (dfil <= fil_interval[1])
nonfil = (dfil >= nonfil_interval[0]) & (dfil <= nonfil_interval[1])
dense = cat[col].values > thr_k

masks = {
    'Full':              np.ones(len(cat), bool),
    'Fil':               fil,
    'NonFil':            nonfil,
    'Fil dense':         fil & dense,
    'Fil underdense':    fil & ~dense,
    'NonFil dense':      nonfil & dense,
    'NonFil underdense': nonfil & ~dense,
}
names = list(masks)
tags = {n: n.replace(' ', '_') for n in names}          # labels in the spectra file
n_gal = {n: int(m.sum()) for n, m in masks.items()}
for n in names:
    print(f"{n:<18} N = {n_gal[n]:>8d}   nbar = {n_gal[n] / L**3:.2e}   "
          f"dense fraction = {np.mean(dense[masks[n]]):.2f}")

## 4. Monopoles

`compute_xi_s_mu` + `compute_monopole_from_xi_s_mu`, the pipeline's functions: DD from Corrfunc,
analytic RR (exact in the periodic box), average over $\mu$. Pair counts are cached in `OUT`;
$N$ is in the file name, so a different sample can never reuse the wrong counts.
The first run takes a few minutes (mostly the Full sample).

In [ ]:
xi0 = {}
for n in names:
    g = cat.loc[masks[n]]
    pc = os.path.join(OUT, f"paircounts_{tags[n]}_N{n_gal[n]}_mag{mag_max:.1f}"
                           f"_sep{min_sep}-{max_sep}_bin{bin_size}.npz")
    xi, s_edges, _ = compute_xi_s_mu(g['x'].values, g['y'].values, g['z'].values,
                                     min_sep, max_sep, bin_size, boxsize=L,
                                     paircounts_filename=pc, force_recompute=force_recompute)
    xi0[n] = compute_monopole_from_xi_s_mu(xi)
s = 0.5 * (s_edges[:-1] + s_edges[1:])
nb = len(s)

fig, ax = plt.subplots(figsize=(7, 4.5))
for n in names:
    ax.plot(s, s**2 * xi0[n], label=n)
ax.set_xlabel(r'$s\ [h^{-1}{\rm Mpc}]$'); ax.set_ylabel(r'$s^2\xi_0$')
ax.legend(fontsize=8); plt.show()

## 5. Joint covariance of all monopoles

The samples share the box (and Fil ⊂ Full, Fil dense ⊂ Fil, ...), so their errors are
correlated. We use `gaussian_cov_pk` from `bao_fitting`, the same joint covariance the fit's
`difference_test` uses for correlated samples: Gaussian covariance of the bin-averaged monopoles
from the **measured** auto- and cross-spectra. The cross-spectra carry both the shared large-scale
modes and, for nested samples, the shared shot noise.

Two details of that function:
* below $k = 0.08$ it replaces the (few, noisy) measured modes by $b_A b_B P_{\rm lin} + N_{AB}$, so it
  needs a bias per sample. As in `run_all`, $b$ comes from `estimate_bias_independent` (30–75 $h^{-1}$Mpc);
  it only shapes the covariance, never the estimate of $R$. Its "unstable" warnings are expected
  for samples whose $\xi$ is not $b^2\xi_{\rm lin}$-shaped, which is exactly what step 8 tests.
* *Not* `multidark_power.xi_covariance_from_spectra` (the script's error-band function): its
  high-$k$ tail fit, $N + A k^{-n}$, is degenerate for cross-spectra of disjoint samples (Fil × NonFil
  goes negative at $k \sim 0.1$–$0.3$, they avoid each other), and returned a plateau of $-2\times10^6$
  that blew up the cross blocks. Auto blocks are fine there, so the script's error bands are unaffected.

We need the cross blocks because the estimator in step 7 uses
$\mathrm{Cov}[\xi_A - R\,\xi_B] = C_{AA} + R^2 C_{BB} - R\,(C_{AB} + C_{BA})$.
Shared sample variance partly cancels in that difference, which makes $R$ more precise than
two independent amplitudes would.

Caveat: Gaussian covariance misses the connected (non-Gaussian) term, which grows towards small
$s$. Errors at $s \lesssim 25\,h^{-1}$Mpc are therefore somewhat underestimated, which makes the
plateau test of step 8 *stricter* there (conservative for the choice of $s_{\min}$).

In [ ]:
pk_file = os.path.join(OUT, f"pk_rho{k_nn}_samples_ng{pk_ngrid}.npz")
need = force_recompute or not os.path.exists(pk_file)
if not need:   # reuse only if it holds exactly these samples
    d = np.load(pk_file, allow_pickle=True)
    need = ([str(x) for x in d['labels']] != [tags[n] for n in names]
            or list(d['n_gal']) != [n_gal[n] for n in names])
if need:
    measure_spectra([cat.loc[masks[n], ['x', 'y', 'z']].values for n in names],
                    [tags[n] for n in names], L, ng=pk_ngrid, save=pk_file)

lin = linear_theory()
k_t, P_w, P_nw = generate_templates(lin=lin)
T = BAOTemplate(k_t, P_w, P_nw, lin)

spec = load_spectra(pk_file)
bias_cov = {n: estimate_bias_independent(s, xi0[n], s_edges, n_gal[n] / L**3, T, L)['b'] for n in names}
print("b used inside the covariance only: " + ", ".join(f"{n} {b:.2f}" for n, b in bias_cov.items()))
C = gaussian_cov_pk(s_edges, spec, [spec['labels'].index(tags[n]) for n in names],
                    [bias_cov[n] for n in names], lin, L)
idx = {n: i for i, n in enumerate(names)}          # blocks of C in the order of `names`

def block(a, b):
    # Cov[xi_a(s_i), xi_b(s_j)]
    i, j = idx[a], idx[b]
    return C[i*nb:(i+1)*nb, j*nb:(j+1)*nb]

sig = {n: np.sqrt(np.diag(block(n, n))) for n in names}

# Sanity check: the joint covariance must be positive definite on the scales we use
use = np.tile(s >= min(s_min_scan), len(names))
ev = np.linalg.eigvalsh(C[np.ix_(use, use)] / np.sqrt(np.outer(np.diag(C)[use], np.diag(C)[use])))
print(f"joint correlation matrix (s >= {min(s_min_scan)}): min eigenvalue {ev.min():.2e} "
      + ("(positive definite)" if ev.min() > 0 else "-> NOT positive definite, check the spectra"))
i50 = np.argmin(np.abs(s - 50))
for a, b in key_pairs:
    r = block(a, b)[i50, i50] / (sig[a][i50] * sig[b][i50])
    print(f"corr(xi_{a}, xi_{b}) at s = {s[i50]:.1f}: {r:+.2f}")

## 6. Upper limit of the scale range: keep the BAO out

If the two samples have *different* BAO amplitudes (the very thing we want to test), any bin
with a sizeable BAO contribution would make $R$ partly measure that difference, which would be
circular. So the bias ratio must come from scales where the BAO contributes little.

From the template of `bao_fitting` (the same Wallisch wiggle/no-wiggle split as the fit, damped
with the theory $\Sigma$, averaged over the same bins), the fractional BAO contribution per bin is

$$f(s) = |\xi_{\rm wig}(s;\Sigma)| \,/\, |\xi_{\rm nw}(s)| .$$

If sample A has a BAO amplitude a fraction $\varepsilon$ larger than B (relative to its bias),
$R$ is biased by about $\varepsilon\,\bar f$, with $\bar f$ the weighted mean of $f$ over the bins used.
With $f \le 5\%$ and even $\varepsilon = 0.5$, that is $\lesssim 2.5\%$ on $R$; compare it with $\sigma_R$ below.

Note that with this no-wiggle definition the BAO part is not just the peak: the whole $r_d$
feature, dip included, is in $\xi_{\rm wig}$, so $f$ is already a few per cent at 50–60 $h^{-1}$Mpc.
Step 8 also scans $s_{\max}$, to show how $R$ moves as the BAO enters.

In [ ]:
Sigma_th = sigma_nl_theory(lin)['Sigma_mono']        # T and lin built in step 5

f_bao = np.abs(T.xi_wig(Sigma_th, edges=s_edges)) / np.abs(T.xi_nw(edges=s_edges))
first_bad = int(np.argmax(f_bao > bao_tolerance))
s_max_suggested = float(s_edges[first_bad])            # upper edge of the last clean bin
S_MAX_USE = S_MAX if S_MAX is not None else s_max_suggested
print(f"Sigma (theory) = {Sigma_th:.2f};  f > {bao_tolerance:.0%} from the bin "
      f"[{s_edges[first_bad]:.0f}, {s_edges[first_bad+1]:.0f}]  ->  s_max = {S_MAX_USE:.0f} h^-1 Mpc")

fig, ax = plt.subplots(figsize=(6.5, 4))
ax.semilogy(s, f_bao, 'o-', ms=3)
ax.axhline(bao_tolerance, color='k', ls='--', lw=1)
ax.axvline(S_MAX_USE, color='tomato', lw=1)
ax.set_xlabel(r'$s\ [h^{-1}{\rm Mpc}]$'); ax.set_ylabel(r'$|\xi_{\rm wig}|/|\xi_{\rm nw}|$')
ax.set_ylim(1e-4, 1e2); ax.set_title('fractional BAO contribution per bin (template)')
plt.show()

## 7. Estimator for $R$

For a trial $R$, the residual $d = \xi_A - R\,\xi_B$ (bins inside $[s_{\min}, s_{\max}]$) has
covariance $C_d(R) = C_{AA} + R^2 C_{BB} - R\,(C_{AB} + C_{BA})$, and

$$\chi^2(R) = d^{\rm T} C_d(R)^{-1} d .$$

We scan $R$ on a fine grid: $\hat R$ = minimum, 1σ interval from $\Delta\chi^2 = 1$, and
$\chi^2_{\min}$ with $n_{\rm bins} - 1$ degrees of freedom tells whether *one constant* describes
the two monopoles over that range.

This is not an ad-hoc choice: it is exactly the $\chi^2$ of the joint data $(\xi_A, \xi_B)$
**profiled over the unknown true $\xi_B$ in every bin** (the model is $\xi_A = R\mu$, $\xi_B = \mu$;
$d$ is the part of the data the free $\mu$ cannot absorb). With noisy $\xi_B$ the profile can drift
to large $R$; the tables flag an interval that reaches the edge of the grid (`*`), and any $R$ at
which $C_d$ is not positive definite (`!`, a sign of a problem in the covariance).

Why this instead of fitting the ratio curve: $\xi_A/\xi_B$ has skewed, hard-to-propagate errors
(it divides by a noisy number), while $d$ is linear in the data and has an exact Gaussian
covariance. Why not $B$ from the BAO fit: $B$ also absorbs the template normalisation and trades
off with the broadband terms; $\hat R$ uses no model at all.

In [ ]:
R_grid = np.logspace(-2, 2, 9211)               # 0.01-100, relative step 0.1 %

def fit_ratio(a, b, s_lo, s_hi):
    # xi_a = R xi_b on the bins fully inside [s_lo, s_hi]
    m = (s_edges[:-1] >= s_lo - 1e-9) & (s_edges[1:] <= s_hi + 1e-9)
    xa, xb = xi0[a][m], xi0[b][m]
    Caa, Cbb, Cab = (block(a, a)[np.ix_(m, m)], block(b, b)[np.ix_(m, m)],
                     block(a, b)[np.ix_(m, m)])
    chi2 = np.full(len(R_grid), np.nan)
    for i, R in enumerate(R_grid):
        d = xa - R * xb
        try:                                   # Cholesky: fails if C_d is not positive definite
            y = np.linalg.solve(np.linalg.cholesky(Caa + R**2 * Cbb - R * (Cab + Cab.T)), d)
            chi2[i] = y @ y
        except np.linalg.LinAlgError:
            pass
    n_bad = int(np.isnan(chi2).sum())
    i0 = int(np.nanargmin(chi2))
    ok = chi2 <= chi2[i0] + 1.0                           # NaN compares False
    lo_i, hi_i = i0, i0                                   # contiguous Delta chi2 = 1 interval
    while lo_i > 0 and ok[lo_i - 1]: lo_i -= 1
    while hi_i < len(R_grid) - 1 and ok[hi_i + 1]: hi_i += 1
    dof = int(m.sum()) - 1
    d1 = xa - xb                                          # R = 1 (equal bias), for the verdict
    chi2_1 = float(d1 @ np.linalg.solve(Caa + Cbb - (Cab + Cab.T), d1))
    z1 = np.sign(R_grid[i0] - 1) * np.sqrt(max(chi2_1 - chi2[i0], 0.0))
    return dict(R=R_grid[i0], z_equal=float(z1), lo=R_grid[lo_i], hi=R_grid[hi_i],
                sigma=0.5 * (R_grid[hi_i] - R_grid[lo_i]),
                chi2=float(chi2[i0]), dof=dof, p=float(stats.chi2.sf(chi2[i0], dof)),
                n_bins=int(m.sum()), s_lo=s_lo, s_hi=s_hi, n_not_pd=n_bad,
                at_edge=(lo_i == 0) or (hi_i == len(R_grid) - 1))

def flags(r):
    # '*' = 68% interval reaches the R grid edge (R unconstrained);  '!' = C_d not pos. def. somewhere
    return ('*' if r['at_edge'] else '') + ('!' if r['n_not_pd'] else '')

## 8. Lower limit: the plateau test

Below a few tens of $h^{-1}$Mpc the bias becomes scale-dependent (non-linear and halo-exclusion
effects; roughly $\propto (R_{\rm halo}/s)^2$), and the coefficients are unknown for these samples.
So we let the data decide: for each comparison, fit a constant $R$ on $[s_{\min}, s_{\max}]$ for
increasing $s_{\min}$. The plateau starts at the smallest $s_{\min}$ for which the constant fits
($p > p_{\rm plateau}$, with $R$ constrained) **and keeps fitting** for every larger $s_{\min}$. A drift of $\hat R$ with
$s_{\min}$ that stops at some scale is the same information, seen by eye.

One common $s_{\min}$ (the largest of the suggested ones) is then used for all comparisons, so
every number in step 9 comes from the same scales.

Second panel: $s_{\max}$ scan at that $s_{\min}$. Beyond the clean limit, a trend in $\hat R$ is
the BAO (or bias scale dependence) entering; it is shown only to see how much it matters, and is
not used for any number.

In [ ]:
comparisons = [(n, reference) for n in names if n != reference] + list(key_pairs)

def suggested_s_min(rows):
    ok = [(r['p'] > p_plateau) and not r['at_edge'] for r in rows]   # an unconstrained R does not count
    for i in range(len(rows)):
        if all(ok[i:]):
            return rows[i]['s_lo']
    return None

scan, sugg = {}, {}
for a, b in comparisons:
    scan[(a, b)] = [fit_ratio(a, b, smin, S_MAX_USE) for smin in s_min_scan
                    if S_MAX_USE - smin >= 3 * bin_size]
    sugg[(a, b)] = suggested_s_min(scan[(a, b)])
    print(f"{a + ' / ' + b:<34} " + "  ".join(
        f"[{r['s_lo']:.0f}: R={r['R']:.3f}±{r['sigma']:.3f}{flags(r)}, p={r['p']:.2f}]" for r in scan[(a, b)])
        + f"   -> plateau from {sugg[(a, b)]}")

found = [v for v in sugg.values() if v is not None]
S_MIN_USE = S_MIN if S_MIN is not None else (max(found) if found else max(s_min_scan))
no_plateau = [k for k, v in sugg.items() if v is None]
print(f"\nCommon range: [{S_MIN_USE}, {S_MAX_USE:.0f}] h^-1 Mpc")
if no_plateau:
    print("NO plateau for: " + ", ".join(f"{a} / {b}" for a, b in no_plateau)
          + "  -> either R is unconstrained (*) or the relative bias is scale-dependent over the"
          " whole scan (see the plots); these do not set the common s_min")

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))
smax_scan = [x for x in np.arange(S_MAX_USE, 105.0, bin_size)]
for c, (a, b) in enumerate(comparisons):
    rows = scan[(a, b)]
    lab = f"{a} / {b}"
    axs[0].errorbar([r['s_lo'] + 0.4 * c for r in rows], [r['R'] for r in rows],
                    yerr=[r['sigma'] for r in rows], fmt='o-', ms=3, capsize=2, label=lab)
    axs[1].plot([r['s_lo'] for r in rows], [r['p'] for r in rows], 'o-', ms=3, label=lab)
    rows2 = [fit_ratio(a, b, S_MIN_USE, smax) for smax in smax_scan]
    R0 = rows2[0]['R']
    axs[2].errorbar([r['s_hi'] + 0.4 * c for r in rows2], [r['R'] / R0 for r in rows2],
                    yerr=[r['sigma'] / R0 for r in rows2], fmt='o-', ms=3, capsize=2, label=lab)
axs[0].set_xlabel(r'$s_{\min}$'); axs[0].set_ylabel(r'$\hat R$'); axs[0].set_title(f'$s_{{\\max}}$ = {S_MAX_USE:.0f}')
axs[1].axhline(p_plateau, color='k', ls='--', lw=1); axs[1].set_yscale('log'); axs[1].set_ylim(1e-6, 2)
axs[1].set_xlabel(r'$s_{\min}$'); axs[1].set_ylabel('p (constant R)')
axs[2].axhline(1, color='k', lw=0.8)
axs[2].set_xlabel(r'$s_{\max}$'); axs[2].set_ylabel(r'$\hat R / \hat R(s_{\max}^{\rm clean})$')
axs[2].set_title(f'$s_{{\\min}}$ = {S_MIN_USE}  (beyond the first point: BAO enters)')
axs[0].legend(fontsize=7)
plt.tight_layout(); plt.show()

The ratio curves themselves, for the key pairs, with $\hat R$ from the common range.
Errors are linear propagation of the joint covariance (diagonal only): **for the eye**, the
numbers come from step 7. Shaded: outside the range used.

In [ ]:
fig, axs = plt.subplots(1, len(key_pairs), figsize=(6 * len(key_pairs), 4), squeeze=False)
mplot = s <= 100
for ax, (a, b) in zip(axs[0], key_pairs):
    xa, xb = xi0[a], xi0[b]
    ca, cb, cab = np.diag(block(a, a)), np.diag(block(b, b)), np.diag(block(a, b))
    ratio = xa / xb
    err = np.abs(ratio) * np.sqrt(np.maximum(ca / xa**2 + cb / xb**2 - 2 * cab / (xa * xb), 0))
    r = fit_ratio(a, b, S_MIN_USE, S_MAX_USE)
    ax.errorbar(s[mplot], ratio[mplot], yerr=err[mplot], fmt='o', ms=3, capsize=2, color='k')
    ax.axhspan(r['lo'], r['hi'], color='steelblue', alpha=0.3, label=rf"$\hat R$ = {r['R']:.3f} ± {r['sigma']:.3f}")
    ax.axvspan(0, S_MIN_USE, color='0.85'); ax.axvspan(S_MAX_USE, 100, color='0.85')
    ax.set_xlim(0, 100)
    lo, hi = np.percentile(ratio[(s >= S_MIN_USE) & (s <= S_MAX_USE)], [0, 100])
    ax.set_ylim(lo - 0.6 * (hi - lo) - 0.1, hi + 0.6 * (hi - lo) + 0.1)
    ax.set_xlabel(r'$s\ [h^{-1}{\rm Mpc}]$'); ax.set_ylabel(rf'$\xi_{{\rm {a}}}/\xi_{{\rm {b}}}$'.replace(' ', r'\ '))
    ax.set_title(f'{a} / {b}'); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## 9. Results

**Bias relative to Full**, $b_X/b_{\rm Full} = \sqrt{R_X}$, all on the common range. This is the
ordering of the samples in large-scale bias, measured without the fit.

**Key pairs.** For NonFil dense / Fil underdense, $\hat R = (b_{\rm NonFil\,dense}/b_{\rm Fil\,under})^2$:
* $\hat R > 1$ significantly: NonFil-dense is *more* biased and still shows less BAO, so BAO
  strength does not follow bias. The strong version of the density-cell result.
* $\hat R$ consistent with 1: equal bias, different BAO. Still says the BAO difference is not a
  bias effect.
* $\hat R < 1$ significantly: Fil-underdense is the more biased sample, so the density cells alone
  cannot separate "more BAO" from "more bias"; the argument then needs the bias-only prediction
  at the BAO scale (next step, not in this notebook).

$z = {\rm sign}(\hat R - 1)\sqrt{\chi^2(R{=}1) - \chi^2_{\min}}$, from the same $\chi^2(R)$, so it stays right
when the interval is very asymmetric (as it is when $\xi_B$ is small). The ± on $b_A/b_B$ is the
half-width of the interval, only indicative.

In [ ]:
res_ref = {n: fit_ratio(n, reference, S_MIN_USE, S_MAX_USE) for n in names if n != reference}
print(f"Range [{S_MIN_USE}, {S_MAX_USE:.0f}] h^-1 Mpc   (real space, Gaussian covariance from measured P(k))\n")
print(f"{'sample':<18}{'R = (b/b_Full)^2':>20}{'b/b_Full':>16}{'chi2/dof':>11}{'p':>7}")
for n, r in sorted(res_ref.items(), key=lambda kv: kv[1]['R']):
    b, sb = np.sqrt(r['R']), r['sigma'] / (2 * np.sqrt(r['R']))
    print(f"{n:<18}{r['R']:>10.3f} [{r['lo']:.3f},{r['hi']:.3f}]{b:>9.3f} ± {sb:.3f}"
          f"{r['chi2']:>6.1f}/{r['dof']:<3d}{r['p']:>7.3f}  {flags(r)}")

print()
res_pairs = {}
for a, b in key_pairs:
    r = fit_ratio(a, b, S_MIN_USE, S_MAX_USE); res_pairs[(a, b)] = r
    z = r['z_equal']
    br, sbr = np.sqrt(r['R']), r['sigma'] / (2 * np.sqrt(r['R']))
    print(f"{a} / {b}:  R = {r['R']:.3f}  (68%: {r['lo']:.3f}-{r['hi']:.3f})   "
          f"b_A/b_B = {br:.3f} ± {sbr:.3f}   z(R=1) = {z:+.2f}   chi2 = {r['chi2']:.1f}/{r['dof']}, p = {r['p']:.3f}  {flags(r)}")
    verdict = ('R not constrained on this range: no conclusion' if r['at_edge'] else
               f'{a} more biased than {b}' if z > 2 else f'{b} more biased than {a}' if z < -2
               else 'consistent with equal bias')
    if r['p'] < p_plateau and not r['at_edge']:
        verdict += f"  (but a constant R does not fit here, p = {r['p']:.3f}: scale-dependent; see step 8)"
    print(f"    -> {verdict}")

key_out = ('R', 'lo', 'hi', 'sigma', 'z_equal', 'chi2', 'dof', 'p')
summary = dict(k_nn=k_nn, dense_percentile=dense_percentile, s_range=[S_MIN_USE, S_MAX_USE],
               bao_tolerance=bao_tolerance, p_plateau=p_plateau, n_gal=n_gal,
               vs_reference={n: {k: r[k] for k in key_out}
                             for n, r in res_ref.items()},
               pairs={f"{a} / {b}": {k: r[k] for k in key_out}
                      for (a, b), r in res_pairs.items()})
with open(os.path.join(OUT, 'summary.json'), 'w') as fh:
    json.dump(summary, fh, indent=1, default=float)
print("\n* = 68% interval reaches the R grid edge;  ! = C_d not positive definite for some R")
print(f"Saved {os.path.join(OUT, 'summary.json')}")

## What this does and does not show

* It measures the **ordering of large-scale bias** on 20–60 $h^{-1}$Mpc-type scales, model-free.
  It does not by itself say how much of the BAO difference bias could explain; that is the
  bias-only prediction ($\xi_A$ vs $\hat R\,\xi_B$ at the peak), a separate step.
* It assumes the relative bias measured below the BAO also holds at ~100 $h^{-1}$Mpc. That is the
  conservative direction (bias becomes more constant at larger scales), but it is an assumption.
* Real space only. In redshift space the monopole ratio picks up $(1 + 2\beta/3 + \beta^2/5)$ factors
  and tracer-dependent small-scale velocities, so for SDSS/Flagship $s_{\min}$ would have to be larger.
* If a comparison has **no plateau**, its relative bias is scale-dependent over the whole range.
  That is a result in itself (worth knowing for NonFil, whose bias estimate in the fit was already
  unstable across 30–75 $h^{-1}$Mpc), and it means that pair cannot be summarised by one number.
* To compare with $\rho_3$: set `k_nn = 3` and rerun (separate output folder).